# 17. Genes on the Genome: Annotation and Intervals

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `hbb_regions.py`

Where in HBB do ClinVar's variants fall, and does that depend on whether they cause disease? Uses the canonical transcript, HBB-201.


In [ ]:
# Where in HBB do ClinVar's variants fall, and does that depend on whether
# they cause disease? Uses the canonical transcript, HBB-201.
import json
import urllib.request
from collections import Counter
from pathlib import Path

import pysam

ENSEMBL = "https://rest.ensembl.org/lookup/id/ENST00000335295"
SAVED = Path("hbb_transcript.json")

if not SAVED.exists():
    request = urllib.request.Request(
        ENSEMBL + "?expand=1;utr=1",
        headers={"Accept": "application/json"},
    )
    with urllib.request.urlopen(request) as response:
        SAVED.write_bytes(response.read())
transcript = json.loads(SAVED.read_text())

# Ensembl counts from 1 and includes both ends; convert every interval to
# Python's convention, zero based with the end excluded, once, here.
exons = [(e["start"] - 1, e["end"]) for e in transcript["Exon"]]
coding = (
    transcript["Translation"]["start"] - 1,
    transcript["Translation"]["end"],
)
gene = (transcript["start"] - 1, transcript["end"])


def inside(position: int, interval: tuple[int, int]) -> bool:
    """Is a zero based position inside a half open interval?"""
    start, end = interval
    return start <= position < end


def region(position: int) -> str:
    if not inside(position, gene):
        return "outside the gene"
    if not any(inside(position, exon) for exon in exons):
        return "intron"
    if inside(position, coding):
        return "coding exon"
    return "untranslated exon"


counts = Counter()
for record in pysam.VariantFile("hbb_clinvar.vcf"):
    significance = record.info.get("CLNSIG", ("none",))[0]
    if significance in ("Pathogenic", "Likely_benign"):
        counts[significance, region(record.pos - 1)] += 1

for significance in ("Pathogenic", "Likely_benign"):
    print(significance.replace("_", " "))
    for place in (
        "coding exon",
        "untranslated exon",
        "intron",
        "outside the gene",
    ):
        print(f"  {place:18} {counts[significance, place]:4}")

## `nearest.py`

For each variant between genes, the nearest gene on each side, and whether the variant lies upstream of that gene's start.


In [ ]:
# For each variant between genes, the nearest gene on each side, and
# whether the variant lies upstream of that gene's start.
from bisect import bisect_right

import pandas as pd

VARIANTS = [281_923, 433_359, 2_333_538, 2_999_330, 3_909_807]  # Ch. 16

columns = [
    "seqid",
    "source",
    "type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes",
]
gff = pd.read_csv(
    "../ch15/rel606.gff3", sep="\t", comment="#", names=columns
)
genes = gff[gff["type"] == "gene"].sort_values("start")
genes = genes.assign(
    name=genes["attributes"].str.extract(r"Name=([^;]+)", expand=False)
)
starts = genes["start"].tolist()


def describe(gene, position: int) -> str:
    """Name, distance, and whether the position is before its start."""
    if gene.end < position:
        distance = position - gene.end
    else:
        distance = gene.start - position
    begins = gene.start if gene.strand == "+" else gene.end
    upstream = (begins > position) == (gene.strand == "+")
    side = "upstream" if upstream else "downstream"
    return f"{gene['name']} ({gene.strand}) {distance} bp, {side}"


for position in VARIANTS:
    # bisect finds where the position would go in the sorted starts
    after = bisect_right(starts, position)
    left = genes.iloc[after - 1]
    while left.end >= position:  # skip any gene that contains it
        after -= 1
        left = genes.iloc[after - 1]
    right = genes.iloc[bisect_right(starts, position)]
    print(f"{position:9,}")
    print(f"  left:  {describe(left, position)}")
    print(f"  right: {describe(right, position)}")